In [65]:
import pandas as pd
import numpy as np


from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier



from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn import svm
from sklearn import neighbors
import xgboost as xgb


In [66]:
#importing data 

#df with dummy categorical variables, no missing data
dummy_df = pd.read_csv("../data/processed/dummy_df_no_missing.csv")
#df with dummy categorical variables, missing data in contact column
dummy_full_contact_df = pd.read_csv("../data/processed/dummy_df_with_contact.csv")
#df with encoded categorical variables, no missing data
encoded_df = pd.read_csv("../data/processed/encoded_df_no_missing.csv")
#df with encoded categorical variables, missing data in contact column
encoded_full_contact_df = pd.read_csv("../data/processed/encoded_df_with_contact.csv")



In [67]:
datasets = {
    "encoded_with_contact": encoded_full_contact_df,
    "encoded": encoded_df,
    "dummy_with_contact": dummy_full_contact_df,
    "dummy": dummy_df,
}

In [68]:
encoded_full_contact_df.describe()

,age,job,marital,education,default,balance,housing,loan,contact,day,duration,campaign,y,month_sin,month_cos
count,38469.000000,38469.000000,38469.000000,38469.000000,38469.000000,38469.000000,38469.000000,38469.000000,38469.000000,38469.000000,38469.000000,38469.000000,38469.000000,3.846900e+04,38469.000000
mean,40.403624,4.259300,1.153578,1.128311,0.020198,1267.535288,0.604305,0.176844,0.686189,16.012686,254.981726,2.875848,0.072786,5.672233e-02,-0.512701
std,9.593403,3.276512,0.606854,0.661691,0.140679,2890.878338,0.489006,0.381541,0.918902,8.261939,259.678378,3.200970,0.259788,5.939143e-01,0.617418
min,19.000000,0.000000,0.000000,0.000000,0.000000,-8019.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,-8.660254e-01,-1.000000
25%,33.000000,1.000000,1.000000,1.000000,0.000000,54.000000,0.000000,0.000000,0.000000,8.000000,100.000000,1.000000,0.000000,-5.000000e-01,-0.866025
50%,39.000000,4.000000,1.000000,1.000000,0.000000,403.000000,1.000000,0.000000,0.000000,17.000000,175.000000,2.000000,0.000000,1.224647e-16,-0.866025
75%,48.000000,7.000000,2.000000,2.000000,0.000000,1314.000000,1.000000,0.000000,2.000000,21.000000,313.000000,3.000000,0.000000,5.000000e-01,-0.500000
max,95.000000,11.000000,2.000000,2.000000,1.000000,102127.000000,1.000000,1.000000,2.000000,31.000000,4918.000000,58.000000,1.000000,1.000000e+00,1.000000


In [69]:
def prepare_data(df):
    # Separate features and target
    X = df.drop(columns="y")
    y = df["y"]

    #Create training and testing datasets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # Synthetic Minority Over-sampling Technique (SMOTE)
    sm = SMOTE(random_state=42)
    X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

    # Random undersampler
    rus = RandomUnderSampler(random_state=42)
    X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

    return {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "X_train_sm": X_train_sm,
        "y_train_sm": y_train_sm,
        "X_train_rus": X_train_rus,
        "y_train_rus": y_train_rus,
    }

In [70]:
results = {}
for name, df in datasets.items():
    results[name] = prepare_data(df)

In [72]:
# Finding F1 with logistic regression
comparison = []
for name, data in results.items():

    # SMOTE model
    model_sm = LogisticRegression(random_state=42, max_iter=2000)
    model_sm.fit(data["X_train_sm"], data["y_train_sm"])
    y_pred_sm = model_sm.predict(data["X_test"])
    f1_sm = f1_score(data["y_test"], y_pred_sm)

    # Random undersampling model
    model_rus = LogisticRegression(random_state=42, max_iter=2000)
    model_rus.fit(data["X_train_rus"], data["y_train_rus"])
    y_pred_rus = model_rus.predict(data["X_test"])
    f1_rus = f1_score(data["y_test"], y_pred_rus)

    comparison.append({
        "Dataset": name,
        "SMOTE F1": f1_sm,
        "RUS F1": f1_rus
    })

print(pd.DataFrame(comparison))


/Users/megmkr/Documents/Apziva/TermDepositMarketing/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/megmkr/Documents/Apziva/TermDepositMarketing/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the dat

                Dataset  SMOTE F1    RUS F1
0  encoded_with_contact  0.436600  0.448864
1               encoded  0.455148  0.463942
2    dummy_with_contact  0.469136  0.479597
3                 dummy  0.424537  0.508794


/Users/megmkr/Documents/Apziva/TermDepositMarketing/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [73]:
"""feature_names = X.columns
coefficients = model.coef_[0]

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Abs_Importance': np.abs(coefficients)
})

importance_df = importance_df.sort_values(by='Abs_Importance', ascending=False).reset_index(drop=True)
print(importance_df)

plt.figure(figsize=(8, 5))
plt.barh(importance_df['Feature'], importance_df['Abs_Importance'], color='skyblue')
plt.xlabel('Feature Importance (Absolute Coefficient)')
plt.ylabel('Features')
plt.title('Logistic Regression Feature Importance')
plt.gca().invert_yaxis()  # Put highest importance on top
plt.show()"""

"feature_names = X.columns\ncoefficients = model.coef_[0]\n\nimportance_df = pd.DataFrame({\n    'Feature': feature_names,\n    'Coefficient': coefficients,\n    'Abs_Importance': np.abs(coefficients)\n})\n\nimportance_df = importance_df.sort_values(by='Abs_Importance', ascending=False).reset_index(drop=True)\nprint(importance_df)\n\nplt.figure(figsize=(8, 5))\nplt.barh(importance_df['Feature'], importance_df['Abs_Importance'], color='skyblue')\nplt.xlabel('Feature Importance (Absolute Coefficient)')\nplt.ylabel('Features')\nplt.title('Logistic Regression Feature Importance')\nplt.gca().invert_yaxis()  # Put highest importance on top\nplt.show()"

In [ ]:
# Finding F1 with random forest classifier
comparison = []
for name, data in results.items():

    # SMOTE model
    model_sm = RandomForestClassifier(n_estimators=100,random_state=42)
    model_sm.fit(data["X_train_sm"], data["y_train_sm"])
    y_pred_sm = model_sm.predict(data["X_test"])
    f1_sm = f1_score(data["y_test"], y_pred_sm)

    # Random undersampling model
    model_rus = RandomForestClassifier(n_estimators=100,random_state=42)
    model_rus.fit(data["X_train_rus"], data["y_train_rus"])
    y_pred_rus = model_rus.predict(data["X_test"])
    f1_rus = f1_score(data["y_test"], y_pred_rus)

    comparison.append({
        "Dataset": name,
        "SMOTE F1": f1_sm,
        "RUS F1": f1_rus
    })

print(pd.DataFrame(comparison))


                Dataset  SMOTE F1    RUS F1
0  encoded_with_contact  0.467641  0.491076
1               encoded  0.491525  0.514523
2    dummy_with_contact  0.456740  0.483390
3                 dummy  0.477146  0.505275


In [ ]:
# Finding F1 with deicison tree classifier
comparison = []
for name, data in results.items():

    # SMOTE model
    model_sm = DecisionTreeClassifier(random_state=42)
    model_sm.fit(data["X_train_sm"], data["y_train_sm"])
    y_pred_sm = model_sm.predict(data["X_test"])
    f1_sm = f1_score(data["y_test"], y_pred_sm)

    # Random undersampling model
    model_rus = DecisionTreeClassifier(random_state=42)
    model_rus.fit(data["X_train_rus"], data["y_train_rus"])
    y_pred_rus = model_rus.predict(data["X_test"])
    f1_rus = f1_score(data["y_test"], y_pred_rus)

    comparison.append({
        "Dataset": name,
        "SMOTE F1": f1_sm,
        "RUS F1": f1_rus
    })

print(pd.DataFrame(comparison))


                Dataset  SMOTE F1    RUS F1
0  encoded_with_contact  0.438819  0.392324
1               encoded  0.494686  0.422333
2    dummy_with_contact  0.419952  0.395778
3                 dummy  0.434701  0.419424


In [74]:
"""importances = grid_search.best_estimator_.feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(feature_importance_df)

plt.figure(figsize=(8, 5))
plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'], color='skyblue')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('Decision Tree Feature Importance (MDI)')
plt.gca().invert_yaxis()  # Puts the highest importance at the top
plt.show()"""

"importances = grid_search.best_estimator_.feature_importances_\n\nfeature_importance_df = pd.DataFrame({\n    'Feature': X.columns,\n    'Importance': importances\n}).sort_values(by='Importance', ascending=False)\n\nprint(feature_importance_df)\n\nplt.figure(figsize=(8, 5))\nplt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'], color='skyblue')\nplt.xlabel('Importance Score')\nplt.ylabel('Features')\nplt.title('Decision Tree Feature Importance (MDI)')\nplt.gca().invert_yaxis()  # Puts the highest importance at the top\nplt.show()"

In [ ]:
# Finding F1 with XGBoost
comparison = []
for name, data in results.items():

    # SMOTE model
    model_sm = xgb.XGBClassifier(random_state=42)
    model_sm.fit(data["X_train_sm"], data["y_train_sm"])
    y_pred_sm = model_sm.predict(data["X_test"])
    f1_sm = f1_score(data["y_test"], y_pred_sm)

    # Random undersampling model
    model_rus = xgb.XGBClassifier(random_state=42)
    model_rus.fit(data["X_train_rus"], data["y_train_rus"])
    y_pred_rus = model_rus.predict(data["X_test"])
    f1_rus = f1_score(data["y_test"], y_pred_rus)

    comparison.append({
        "Dataset": name,
        "SMOTE F1": f1_sm,
        "RUS F1": f1_rus
    })

print(pd.DataFrame(comparison))


                Dataset  SMOTE F1    RUS F1
0  encoded_with_contact  0.503441  0.488235
1               encoded  0.519954  0.518430
2    dummy_with_contact  0.519031  0.487591
3                 dummy  0.509128  0.522370


In [ ]:
# Finding F1 with SVM
comparison = []
for name, data in results.items():

    # SMOTE model
    model_sm = svm.SVC(random_state=42)
    model_sm.fit(data["X_train_sm"], data["y_train_sm"])
    y_pred_sm = model_sm.predict(data["X_test"])
    f1_sm = f1_score(data["y_test"], y_pred_sm)

    # Random undersampling model
    model_rus = svm.SVC(random_state=42)
    model_rus.fit(data["X_train_rus"], data["y_train_rus"])
    y_pred_rus = model_rus.predict(data["X_test"])
    f1_rus = f1_score(data["y_test"], y_pred_rus)

    comparison.append({
        "Dataset": name,
        "SMOTE F1": f1_sm,
        "RUS F1": f1_rus
    })

print(pd.DataFrame(comparison))


                Dataset  SMOTE F1    RUS F1
0  encoded_with_contact  0.403475  0.411735
1               encoded  0.429100  0.429026
2    dummy_with_contact  0.403475  0.411940
3                 dummy  0.428660  0.429299


In [ ]:
# Finding F1 with KNN
comparison = []
for name, data in results.items():

    # SMOTE model
    model_sm = neighbors.KNeighborsClassifier()
    model_sm.fit(data["X_train_sm"], data["y_train_sm"])
    y_pred_sm = model_sm.predict(data["X_test"])
    f1_sm = f1_score(data["y_test"], y_pred_sm)

    # Random undersampling model
    model_rus = neighbors.KNeighborsClassifier()
    model_rus.fit(data["X_train_rus"], data["y_train_rus"])
    y_pred_rus = model_rus.predict(data["X_test"])
    f1_rus = f1_score(data["y_test"], y_pred_rus)

    comparison.append({
        "Dataset": name,
        "SMOTE F1": f1_sm,
        "RUS F1": f1_rus
    })

print(pd.DataFrame(comparison))


                Dataset  SMOTE F1    RUS F1
0  encoded_with_contact  0.326345  0.333597
1               encoded  0.336988  0.350249
2    dummy_with_contact  0.327594  0.332673
3                 dummy  0.339338  0.348387


After the preliminary tests, on all four datasets and two sampling models:
* Removing the missing data from the contact column (although it is a larger portion, seems to give more accurate results)
* Using the under sampling method tends to be better than using the oversampling method
* Our top performing model is Random Forest Classifier

In [75]:
"""dt_clf = DecisionTreeClassifier(random_state=42)
param_grid = {
    'criterion': ['gini', 'entropy','log_loss'],
    'max_depth': [5, 7, 10],
    'min_samples_split': [1, 2, 3],
    'min_samples_leaf': [1, 2, 3],
    'min_weight_fraction_leaf' : [None, 0.01,0.03 ,0.05, 0.1, 0.3]
}

grid_search = GridSearchCV(
    estimator=dt_clf, 
    param_grid=param_grid, 
    cv=5, 
    scoring='f1', 
    n_jobs=-1
)
grid_search.fit(X_train, y_train)
y_pred = grid_search.best_estimator_.predict(X_test)
print("\nf1 score:\n", f1_score(y_test, y_pred))
print(grid_search.best_params_)"""

'dt_clf = DecisionTreeClassifier(random_state=42)\nparam_grid = {\n    \'criterion\': [\'gini\', \'entropy\',\'log_loss\'],\n    \'max_depth\': [5, 7, 10],\n    \'min_samples_split\': [1, 2, 3],\n    \'min_samples_leaf\': [1, 2, 3],\n    \'min_weight_fraction_leaf\' : [None, 0.01,0.03 ,0.05, 0.1, 0.3]\n}\n\ngrid_search = GridSearchCV(\n    estimator=dt_clf, \n    param_grid=param_grid, \n    cv=5, \n    scoring=\'f1\', \n    n_jobs=-1\n)\ngrid_search.fit(X_train, y_train)\ny_pred = grid_search.best_estimator_.predict(X_test)\nprint("\nf1 score:\n", f1_score(y_test, y_pred))\nprint(grid_search.best_params_)'